In [1]:
import joblib
import pandas as pd
import numpy as np
from collections import Counter

# Models
svm = joblib.load("svm.pkl")   # optional (only if retrained with probability=True)
lgb = joblib.load("lgb.pkl")
rf = joblib.load("rf.pkl")
xgb = joblib.load("xgb.pkl")

# Preprocessing
scaler = joblib.load("scaler.pkl")
le = joblib.load("label_encoder.pkl")
x_columns = joblib.load("columns.pkl")

# Model comparison (ranking)
all_PM = joblib.load("model_ranking.pkl")

In [2]:
def create_df4_from_df(df, amplitude, condition, level, windows=30):

    def extract_features(arr):
        return {
            "min": np.min(arr),
            "max": np.max(arr),
            "mean": np.mean(arr),
            "std": np.std(arr),
            "var": np.var(arr),
            "rms": np.sqrt(np.mean(arr**2)),
            "mav": np.mean(np.abs(arr))
        }

    data = df.iloc[:, :24].values

    # Magnitude conversion (24 → 8)
    magnitude = []
    for i in range(0, 24, 3):
        x, y, z = data[:, i], data[:, i+1], data[:, i+2]
        mag = np.sqrt(x**2 + y**2 + z**2)
        magnitude.append(mag)

    magnitude = np.array(magnitude).T

    window_size = len(magnitude) // windows
    df4_list = []

    for w in range(windows):
        start = w * window_size
        end = (w + 1) * window_size
        window = magnitude[start:end]

        features = {}
        for i in range(8):
            f = extract_features(window[:, i])
            for k, v in f.items():
                features[f"M{i+1}_{k}"] = v

        features.update({
            "Amplitude": amplitude,
            "Condition": condition,
            "Level": level
        })

        df4_list.append(features)

    return pd.DataFrame(df4_list)

In [3]:
# ================= INPUT FORMAT =================
# amplitude → 0.5, 1, 2
# condition → "Healthy", "6Nm", "9Nm", "NoBolt"
# level     → 0, 1, 2, 3, 4
# ================================================

df_new = pd.read_csv("experiment_8-1.csv")

df4_new = create_df4_from_df(
    df=df_new,
    amplitude=0.5,
    condition="6Nm",
    level=1
)

df4_new.head()

,M1_min,M1_max,M1_mean,M1_std,M1_var,M1_rms,M1_mav,M2_min,M2_max,M2_mean,...,M8_min,M8_max,M8_mean,M8_std,M8_var,M8_rms,M8_mav,Amplitude,Condition,Level
0,0.000284,0.000712,0.000507,0.000067,4.502269e-09,0.000511,0.000507,0.000193,0.000437,0.000316,...,0.000121,0.000334,0.000227,0.00003,9.175367e-10,0.000229,0.000227,0.5,6Nm,1
1,0.000257,0.000753,0.000506,0.000067,4.500945e-09,0.000511,0.000506,0.000174,0.000473,0.000320,...,0.000120,0.000343,0.000226,0.00003,9.183215e-10,0.000228,0.000226,0.5,6Nm,1
2,0.000278,0.000749,0.000506,0.000068,4.571965e-09,0.000510,0.000506,0.000185,0.000442,0.000320,...,0.000128,0.000335,0.000224,0.00003,8.705303e-10,0.000226,0.000224,0.5,6Nm,1
3,0.000287,0.000714,0.000505,0.000066,4.396851e-09,0.000509,0.000505,0.000180,0.000439,0.000319,...,0.000095,0.000311,0.000222,0.00003,8.703221e-10,0.000224,0.000222,0.5,6Nm,1
4,0.000286,0.000708,0.000504,0.000067,4.551201e-09,0.000508,0.000504,0.000189,0.000439,0.000315,...,0.000107,0.000329,0.000222,0.00003,9.068135e-10,0.000224,0.000222,0.5,6Nm,1


In [4]:
# Remove non-feature columns
X_new = df4_new.drop(columns=["Condition", "Level"], errors="ignore")

# Safe column alignment
X_new = X_new.reindex(columns=x_columns, fill_value=0)

# Scale
X_scaled = scaler.transform(X_new)

In [5]:
# Select top 3 models automatically
top_models = all_PM["Model"].head(4).values

# Map names → actual models
model_dict = {
    "SVM": svm,
    "LightGBM": lgb,
    "Random Forest": rf,
    "XGBoost": xgb
}

models = [model_dict[name] for name in top_models]
print("Selected Models:", top_models)

Selected Models: ['SVM' 'LightGBM' 'Random Forest' 'XGBoost']


In [6]:
# Use F1 Score as weight
weights_dict = all_PM.set_index("Model")["F1 Score"].to_dict()
weights = [weights_dict[name] for name in top_models]

# Weighted probability
probs = [m.predict_proba(X_scaled) * w for m, w in zip(models, weights)]
avg_prob = sum(probs) / sum(weights)

# Final prediction
pred = np.argmax(avg_prob, axis=1)
confidence = np.max(avg_prob, axis=1)

final_pred = le.inverse_transform(pred)

C:\Users\lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [7]:
# Window-wise prediction
print("Window Predictions:")
print(final_pred)

# Final condition (majority voting)
final_condition = Counter(final_pred).most_common(1)[0][0]

print("\nFinal Predicted Condition:", final_condition)

# Confidence
final_confidence = np.mean(confidence) * 100
print(f"Confidence: {final_confidence:.2f}%")

Window Predictions:
['6Nm_L1' '6Nm_L1' '6Nm_L1' '6Nm_L1' '6Nm_L1' '6Nm_L1' '6Nm_L1' '6Nm_L1'
 '6Nm_L1' '6Nm_L1' '6Nm_L1' '6Nm_L1' '6Nm_L1' '6Nm_L1' '6Nm_L1' '6Nm_L1'
 '6Nm_L1' '6Nm_L1' '6Nm_L1' '6Nm_L1' '6Nm_L1' '6Nm_L1' '6Nm_L1' '6Nm_L1'
 '6Nm_L1' '6Nm_L1' '6Nm_L1' '6Nm_L1' '6Nm_L1' '6Nm_L1']

Final Predicted Condition: 6Nm_L1
Confidence: 95.21%
